# Approach 3 — Qwen3-VL with LoRA

Notebook 09 is the single maintained training path for the Qwen3-VL baseline and QLoRA adapter. This companion notebook reloads the saved results and allows report review; it intentionally does not duplicate training or start an unbounded run.

## Before running

Use the same Drive account and mount it below. The saved adapter lives under `qwen3vl_cxr/qwen3vl-cxr-lora`; the evaluation CSVs are small. The base model itself is downloaded on demand by notebook 09 or the Docker API. BLEU and ROUGE-L are text overlap metrics, not medical accuracy.

In [1]:
from pathlib import Path
import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
CANDIDATES = [Path.cwd(), Path.cwd().parent, Path("/content/drive/MyDrive/cxr_second_reader"), Path("/content/cxr_second_reader")]
REPO_ROOT = next((p.resolve() for p in CANDIDATES if (p / "src" / "data.py").exists()), None)
if REPO_ROOT is None and Path("/content").exists():
    import subprocess
    clone_dir = Path("/content/cxr_second_reader")
    if not clone_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/eyadstein/cxr_second_reader.git", str(clone_dir)], check=True)
    if (clone_dir / "src" / "data.py").exists():
        REPO_ROOT = clone_dir.resolve()
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook from the cxr_second_reader project checkout, or make the updated project available in Drive/GitHub.")
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print("Project:", REPO_ROOT)

Project: D:\cxr_second_reader


In [2]:
import json
import pandas as pd
from IPython.display import display

DRIVE_RESULTS = Path("/content/drive/MyDrive/qwen3vl_cxr")
LOCAL_RESULTS = REPO_ROOT / "qwen3vl_cxr"
required = ["metrics_qwen_zero_shot.json", "metrics_qwen_lora_finetuned.json",
            "eval_qwen_zero_shot.csv", "eval_qwen_lora_finetuned.csv"]
RESULTS = next((candidate for candidate in [DRIVE_RESULTS, LOCAL_RESULTS]
                if all((candidate / name).is_file() for name in required)), None)
if RESULTS is None:
    raise FileNotFoundError("Saved Qwen metrics and CSVs are missing from both Drive and the project artifacts. Run notebook 09 first.")
print("Results folder:", RESULTS)
base = json.loads((RESULTS / required[0]).read_text(encoding="utf-8"))
ft = json.loads((RESULTS / required[1]).read_text(encoding="utf-8"))
display(pd.DataFrame([{"stage":"Qwen zero-shot", **base}, {"stage":"Qwen QLoRA", **ft}]))

Results folder: D:\cxr_second_reader\qwen3vl_cxr
            stage    rougeL      bleu   n
0  Qwen zero-shot  0.187889  0.026596  24
1      Qwen QLoRA  0.331241  0.069996  24


In [3]:
zero = pd.read_csv(RESULTS / "eval_qwen_zero_shot.csv")
finetuned = pd.read_csv(RESULTS / "eval_qwen_lora_finetuned.csv")
print("Rows:", len(zero), len(finetuned))
print("Matching study IDs:", set(zero["uid"].astype(str)) == set(finetuned["uid"].astype(str)))
review = finetuned[[c for c in ["uid", "filename", "prediction", "reference", "rougeL", "bleu"] if c in finetuned.columns]]
display(review.head(5))
print("Review generated findings against each image; text metrics cannot identify unsupported or omitted clinical findings.")

Rows: 24 24
Matching study IDs: True
    uid                   filename  ...    rougeL      bleu
0  2924  2924_IM-1327-1001.dcm.png  ...  0.313253  0.030792
1  1410  1410_IM-0260-1001.dcm.png  ...  0.356164  0.049788
2  3062  3062_IM-1428-1001.dcm.png  ...  0.326531  0.034173
3   100   100_IM-0002-1001.dcm.png  ...  0.484848  0.110628
4  1219  1219_IM-0146-1001.dcm.png  ...  0.360000  0.075654

[5 rows x 6 columns]
Review generated findings against each image; text metrics cannot identify unsupported or omitted clinical findings.


## Continue training

For more training, return to `09_qwen3vl_cxr.ipynb`, restore the QLoRA checkpoint and optimizer state, and run another saved training chunk. After choosing a checkpoint with validation data, rerun the notebook's final adapter-save and held-out evaluation cells. Do not repeatedly tune against the test split.